# MatSharp: Generic Math scalars
Embedding C# notebook: it runs MATLAB source with MatEngine, it is not a MATLAB Jupyter kernel. The .NET 11 runtime is required for IEEE Decimal. Scalar storage is distinct from the matrices' double buffer.

In [ ]:
#r "D:/Dev/BuildArtifacts/MatSharp/bin/MatSharpLib/release/MatSharpLib.dll"
using MatSharpLib; using MatSharpLib.Runtime;
using System;
var engine = new MatEngine();
foreach (var type in new[] {"decimal32","decimal64","decimal128"}) {
 var value=engine.Eval($"{type}(2) + {type}(3) == {type}(5)");
 if(value is not Mat logical || logical.Re[0]!=1) throw new Exception(type);
 Console.WriteLine(type+"_operators: OK");
}
engine.Run("x=decimal128('1.0000000000000000000000001'); y=decimal128('1e-25');");
var check=(Mat)engine.Eval("x - 1 == y");
if(check.Re[0]!=1)throw new Exception("Precision lost");
Console.WriteLine("matsharp_decimal128_increment: OK");
Console.WriteLine(engine.GetVariable("x")!.ClassName);
try { engine.Eval("decimal128(0.1)"); throw new Exception("Float accepted"); } catch(MatException) { Console.WriteLine("fractional_double_rejected: OK"); }

decimal32_operators: OK
decimal64_operators: OK
decimal128_operators: OK
matsharp_decimal128_increment: OK
decimal128
fractional_double_rejected: OK

## Limits
Typed real 2D scalars and matrices; matrix solve/division, complex numbers and higher dimensions are not implemented. The Mat matrices remain column-major double. Fractional literals must be passed as strings; integers already rounded by the double parser cannot be recovered. No generic CLR binding of the language and no updated package.

## Typed real matrices
Concatenation, transposition, product, indexing and assignment preserve T. MATLAB broadcasting on the leading axes. Real 2D matrices only; no typed or complex solver.

In [ ]:
using System.Numerics;
engine.Run("A=[decimal128(1) decimal128(2); decimal128(3) decimal128(4)]; B=A.'; C=A*B;");
var c=(GenericScalar<Decimal128>)engine.GetVariable("C")!;
var expected=new Decimal128[] {Decimal128.CreateChecked(5),Decimal128.CreateChecked(11),Decimal128.CreateChecked(11),Decimal128.CreateChecked(25)};
if(!c.ToArray().SequenceEqual(expected))throw new Exception("Matrix product mismatch");
Console.WriteLine("typed_matrix_product: OK");
engine.Run("copy=A; A(2,1)=decimal128('1e-25'); v=A(2,1); original=copy(2,1);");
var v=(GenericScalar<Decimal128>)engine.GetVariable("v")!;
if(v.Value!=Decimal128.Parse("1e-25"))throw new Exception("Assignment lost precision");
var original=(GenericScalar<Decimal128>)engine.GetVariable("original")!;
if(original.Value!=Decimal128.CreateChecked(3))throw new Exception("Copy on write lost");
Console.WriteLine("typed_index_assignment_and_copy: OK");

typed_matrix_product: OK
typed_index_assignment_and_copy: OK

In [ ]:
#r "D:/Dev/BuildArtifacts/MathemaSharp/bin/MathemaSharp.LinearAlgebra/release/MathemaSharp.LinearAlgebra.dll"
#r "D:/Dev/BuildArtifacts/MatSharp/bin/MatSharpLib.NDSharp/release/MatSharpLib.NDSharp.dll"
MatSharpLib.NDSharpBridge.NDSharpFunctions.Register(engine);
engine.Run("aa=[decimal128(1) decimal128(0);decimal128(0) decimal128(1)]; bb=[decimal128('1.0000000000000000000000001');decimal128(2)]; xx=aa\\bb;");
var precise = (GenericScalar<Decimal128>)engine.Eval("xx(1)-decimal128(1)");
if (precise.Value != Decimal128.Parse("1e-25")) throw new Exception("Typed solver lost precision");
Console.WriteLine("typed_left_divide_precision: OK");

typed_left_divide_precision: OK